# Exercise 1.9 — Geodesic Length and the Quantum Speed Limit

**Chapter 1: Mathematical Foundations** &nbsp;|&nbsp; *Section 1.7: Riemannian Geometry of Unitary Groups*

---

## Motivation

How fast can a quantum state evolve?  The **quantum speed limit** (Mandelstam-Tamm bound) sets a fundamental lower bound on the time needed to evolve between orthogonal states.  This exercise derives the bound from the geometry of the unitary group: Hamiltonian evolution traces a **geodesic** on $U(D)$ whose speed is set by the energy spread $\Delta H$.  The connection between Riemannian geometry and quantum dynamics is one of the deepest structural insights in quantum information.

## Exercise Statement

In the bi-invariant HS metric on $U(D)$, the geodesic $U(t) = \exp(-iHt)$ has constant speed $\|dU/dt \cdot U^\dagger\| = \sqrt{\mathrm{Tr}(H^2)}$.

**(a)** Compute the geodesic length from $I$ to $U(T) = \exp(-iHT)$.

**(b)** For $H = (\omega/2)\sigma_x$ on a qubit, compute the speed and the length to $U(\pi/\omega) = -i\sigma_x$.

**(c)** Mandelstam-Tamm bound: for $|\psi_0\rangle = |0\rangle$, compute $\Delta H$, predict $T_\perp$, and verify the survival probability vanishes at $t = T_\perp$.

## Solution

### Part (a): Geodesic length

The velocity along the geodesic is:

$$
\frac{dU}{dt}\cdot U^\dagger = (-iH)e^{-iHt} \cdot e^{+iHt} = -iH.
$$

The HS norm on $\mathfrak{u}(D)$ is $\|A\| = \sqrt{-\mathrm{Tr}(A^2)}$ for skew-Hermitian $A$.  Since $-iH$ is skew-Hermitian:

$$
v = \|-iH\| = \sqrt{-\mathrm{Tr}((-iH)^2)} = \sqrt{\mathrm{Tr}(H^2)}.
$$

The speed is **constant** (independent of $t$), so the geodesic length is simply:

$$
\boxed{L = T \cdot \sqrt{\mathrm{Tr}(H^2)}.}
$$

### Part (b): Single-qubit Rabi oscillation

For $H = (\omega/2)\sigma_x$:

$$
\mathrm{Tr}(H^2) = \frac{\omega^2}{4}\mathrm{Tr}(\sigma_x^2) = \frac{\omega^2}{4}\cdot 2 = \frac{\omega^2}{2}.
$$

$$
v = \frac{\omega}{\sqrt{2}}.
$$

At $T = \pi/\omega$ (a $\pi$-rotation about the $x$-axis):

$$
L = \frac{\pi}{\omega} \cdot \frac{\omega}{\sqrt{2}} = \frac{\pi}{\sqrt{2}}.
$$

### Part (c): Mandelstam-Tamm bound

The bound states that the minimum time to evolve to an orthogonal state is:

$$
T_\perp = \frac{\pi}{2\Delta H}, \qquad \Delta H = \sqrt{\langle H^2\rangle - \langle H\rangle^2}.
$$

**Step 1:** Compute the expectation values for $|\psi_0\rangle = |0\rangle$:

$$
\langle H \rangle = \frac{\omega}{2}\langle 0|\sigma_x|0\rangle = 0, \qquad \langle H^2\rangle = \frac{\omega^2}{4}\langle 0|\sigma_x^2|0\rangle = \frac{\omega^2}{4}.
$$

$$
\Delta H = \frac{\omega}{2}.
$$

**Step 2:** Predict $T_\perp$:

$$
T_\perp = \frac{\pi}{2 \cdot \omega/2} = \frac{\pi}{\omega}.
$$

**Step 3:** Verify the survival probability vanishes:

$$
|\langle 0| e^{-iHt}|0\rangle|^2 = |\langle 0| e^{-i(\omega/2)\sigma_x t}|0\rangle|^2 = \cos^2\!\left(\frac{\omega t}{2}\right).
$$

At $t = T_\perp = \pi/\omega$: $\cos^2(\pi/2) = 0$. $\quad \checkmark$

The bound is **saturated** because $|0\rangle$ is an equal superposition of the eigenstates of $\sigma_x$, giving maximal energy spread.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp

w, t = sp.symbols('omega t', real=True, positive=True)
sx = sp.Matrix([[0, 1], [1, 0]])

H = w/2 * sx
trH2 = sp.trace(H * H)
speed = sp.sqrt(trH2)
print(f"Tr(H²) = {trH2}, speed = {sp.simplify(speed)}")

# Survival probability
psi0 = sp.Matrix([1, 0])
expH = sp.exp(-sp.I * H * t)  # matrix exponential
# For σ_x: exp(-i(ω/2)σ_x t) = cos(ωt/2)I - i sin(ωt/2)σ_x
U_t = sp.cos(w*t/2)*sp.eye(2) - sp.I*sp.sin(w*t/2)*sx
psi_t = U_t * psi0
surv = sp.simplify(sp.Abs(psi0.dot(psi_t))**2)
print(f"Survival probability = {surv}")

# At t = π/ω
surv_at_Tperp = surv.subs(t, sp.pi/w)
print(f"At T_⊥ = π/ω: P = {sp.simplify(surv_at_Tperp)}")

---
## Numerical Verification

In [ ]:
import numpy as np
from scipy.linalg import expm

omega = 2.0
sx = np.array([[0,1],[1,0]], dtype=complex)
H = omega/2 * sx
psi0 = np.array([1, 0], dtype=complex)

T_perp = np.pi / omega
delta_H = omega / 2

print(f"ω = {omega}, ΔH = {delta_H}, T_⊥ = π/ω = {T_perp:.4f}")
print(f"Speed = ω/√2 = {omega/np.sqrt(2):.4f}")
print(f"Length at T_⊥ = π/√2 = {np.pi/np.sqrt(2):.4f}")
print()

times = np.linspace(0, 2*T_perp, 9)
print(f"{'t/T⊥':>8s}  {'|⟨0|ψ(t)⟩|²':>14s}")
print(f"{'─'*8:>8s}  {'─'*14:>14s}")
for t_val in times:
    U = expm(-1j * H * t_val)
    p = abs(psi0.conj() @ U @ psi0)**2
    print(f"{t_val/T_perp:8.3f}  {p:14.6f}")

# Check orthogonality at T_perp
U_perp = expm(-1j * H * T_perp)
surv = abs(psi0.conj() @ U_perp @ psi0)**2
assert surv < 1e-10, f"Survival = {surv}"
print(f"\nSurvival at T_⊥ = {surv:.2e}: bound saturated. ✓")